In [61]:
from dotenv import load_dotenv
load_dotenv()

True

In [62]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [63]:
loader =PyPDFLoader("../notebooks/data/medical_report.pdf")
docs=loader.load()

In [64]:
len(docs)

41

In [65]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs =splitter.split_documents(docs)

In [66]:
len(splitted_docs)

121

In [67]:
embeddings =OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [68]:
same_record=vector_store.similarity_search("Patient name")

In [69]:
same_record[0].page_content

'for document retrieval.\nClinical Field\nRecorded Value\nPatient ID\nPAT-10014\nAge\n23\nVisit Type\npreventive care visit\nBlood Pressure\n112/76 mmHg\nHeart Rate\n77 bpm\nTemperature\n97.7 °F\nHemoglobin\n12.5 g/dL\nFasting Glucose\n114 mg/dL\nTotal Cholesterol\n155 mg/dL\nBMI\n19.6\nAssessment\ncontinue preventive care'

In [70]:
## agent = tools, llm, prompt

In [73]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF documents , and these 
        PDF documents have details about medical reports.
    
    """

    print("Tool Called: ",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"

    return context

In [74]:
retriever_tool.invoke("Patient Name")

Tool Called:  Patient Name


'for document retrieval.\nClinical Field\nRecorded Value\nPatient ID\nPAT-10026\nAge\n47\nVisit Type\nroutine health evaluation\nBlood Pressure\n113/86 mmHg\nHeart Rate\n77 bpm\nTemperature\n98.1 °F\nHemoglobin\n15.5 g/dL\nFasting Glucose\n82 mg/dL\nTotal Cholesterol\n146 mg/dL\nBMI\n24.5\nAssessment\nfollow the documented care plan\n\n'

In [ ]:
llm =ChatOpenAI(model="gpt-5")

In [75]:
System_prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    Always use the 'retriever_tool' tool for questions requiring external knowledge.
"""

In [ ]:
agent =create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_prompt
)

In [76]:
query = "what is the name of patient, what is the name of doctors"
response=agent.invoke({"messages":[{"role":"user","content":query}]})

Tool Called:  patient name doctor name
Tool Called:  patient name
Tool Called:  doctor name
Tool Called:  physician name
Tool Called:  Name of Patient
Tool Called:  Patient Name
Tool Called:  Doctor Name
Tool Called:  Provider Name
Tool Called:  Attending Physician
Tool Called:  Name
Tool Called:  Patient Name:
Tool Called:  Doctor:
Tool Called:  Physician:
Tool Called:  Provider:
Tool Called:  Attending:
Tool Called:  Dr.
Tool Called:  Signed by
Tool Called:  Signature
Tool Called:  Patient Name full name
Tool Called:  Name of Patient:
Tool Called:  Doctor Name full name
Tool Called:  Primary Care Physician name
Tool Called:  Referring Physician name
Tool Called:  Consulting Physician name
Tool Called:  Examined by
Tool Called:  Patient Name
Tool Called:  Name:
Tool Called:  Doctor
Tool Called:  Physician Name
Tool Called:  Provider Name
Tool Called:  Attending Physician Name
Tool Called:  Examined by


In [77]:
result =response["messages"][-1].content

In [78]:
print(result)

I couldn’t find any patient or doctor names in the retrieved records—only patient IDs (e.g., PAT-10004, PAT-10016, etc.) and clinical values. If you can specify the exact report (title/date) or share the document, I can look up the names.
